# Walkthrough final: nga dataset-i te prediction-i

Ky notebook përmbledh rrjedhën e projektit duke përdorur modulet dhe output-et ekzistuese. Ai **nuk ritrajnon modelin**, nuk ndryshon calibration-in dhe nuk ndryshon pragjet finale `0.30/0.70`.

> Rezultati është probabilitet sipas modelit dhe karakteristikave të tekstit. Sistemi nuk bën verifikim faktik të lajmit.

In [ ]:
from __future__ import annotations

import ast
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('Hape notebook-un nga rrënja e projektit ose nga notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.linguistic_features import extract_linguistic_features
from src.models.predict_final import (
    FINAL_FAKE_THRESHOLD,
    FINAL_MODEL_ID,
    FINAL_MODEL_PATH,
    FINAL_MODEL_VERSION,
    FINAL_REAL_THRESHOLD,
    load_final_model,
    predict_final_news,
)
from src.preprocessing.clean_text import combine_title_content

pd.set_option('display.max_colwidth', 100)
print(f'Project root: {PROJECT_ROOT}')

## 1. Ngarkimi dhe kontrolli bazë i dataset-it

`src/data/load_dataset.py` lexon skedarët origjinalë dhe `src/data/build_dataset.py` krijon output-in e verifikuar. Për një walkthrough të shpejtë ngarkojmë `articles.parquet`, pa rilexuar 3,994 skedarë raw. Label-et janë `0=real`, `1=fake`.

In [ ]:
processed_dataset_path = PROJECT_ROOT / 'data' / 'processed' / 'articles.parquet'
if not processed_dataset_path.exists():
    raise FileNotFoundError(
        'Mungon articles.parquet. Ekzekuto: python src/data/build_dataset.py'
    )
articles = pd.read_parquet(processed_dataset_path)

class_summary = (
    articles.groupby(['label', 'label_name'], as_index=False)
    .agg(article_count=('article_id', 'count'), unique_pairs=('pair_id', 'nunique'))
)
display(articles[['article_id', 'pair_id', 'label_name', 'title']].head(3))
display(class_summary)
print(f'Total articles: {len(articles):,}')

## 2. Preprocessing bazë

Funksioni i përbashkët normalizon Unicode në NFC, rregullon hapësirat dhe bashkon titullin me përmbajtjen. Nuk heq pikësimin, shkronjat e mëdha ose shkronjat shqipe `ë/ç`.

In [ ]:
sample_article = articles.iloc[0]
sample_model_text = combine_title_content(sample_article['title'], sample_article['content'])

display(pd.DataFrame([{
    'article_id': sample_article['article_id'],
    'title': sample_article['title'],
    'model_text_preview': sample_model_text[:300],
    'model_text_characters': len(sample_model_text),
}]))

## 3. Word + Character TF-IDF

Modeli final përmban dy degë TF-IDF të trajnuara më parë: word n-grams `(1, 2)` dhe character n-grams `(3, 5)`. Këtu vetëm transformojmë një shembull me vectorizer-at e ngrirë; nuk bëhet `fit`.

In [ ]:
final_model = load_final_model(FINAL_MODEL_PATH)
fitted_pipeline = final_model.calibrated_classifiers_[0].estimator
feature_union = fitted_pipeline.named_steps['features']
classifier = fitted_pipeline.named_steps['classifier']
vectorizers = dict(feature_union.transformer_list)

combined_vector = feature_union.transform([sample_model_text])
word_vector = vectorizers['word'].transform([sample_model_text])
word_names = vectorizers['word'].get_feature_names_out()
word_values = word_vector.toarray().ravel()
top_indices = np.argsort(word_values)[-10:][::-1]
top_indices = [index for index in top_indices if word_values[index] > 0]

display(pd.DataFrame([{
    'model_id': FINAL_MODEL_ID,
    'version': FINAL_MODEL_VERSION,
    'branches': ', '.join(vectorizers),
    'vector_shape': str(combined_vector.shape),
    'nonzero_features': int(combined_vector.nnz),
    'classifier': type(classifier).__name__,
    'C': classifier.C,
}]))
display(pd.DataFrame({
    'word_term': word_names[top_indices],
    'tfidf_weight': word_values[top_indices],
}))

## 4. Karakteristikat gjuhësore

Këto karakteristika përshkruajnë formën e tekstit. Në aplikacionin final përdoren **vetëm për shpjegim** dhe nuk ndryshojnë prediction-in e Word + Character TF-IDF + Linear SVM.

In [ ]:
linguistic_features = extract_linguistic_features(
    sample_article['title'], sample_article['content']
)
feature_names = [
    'word_count', 'sentence_count', 'exclamation_count',
    'uppercase_char_ratio', 'diacritic_ratio', 'sensational_found',
    'source_indicators_found', 'uncertainty_found',
]
display(pd.DataFrame({
    'feature': feature_names,
    'value': [linguistic_features[name] for name in feature_names],
}))

## 5. Prediction-et e demonstrimit

Rastet vijnë nga test set-i i ngrirë dhe përputhen me rezultatet zyrtare të Ditës 17. Tre rastet e para formojnë demonstrimin kryesor; false positive dhe false negative shërbejnë për të shpjeguar kufizimet.

In [ ]:
frozen_demos = pd.read_csv(
    PROJECT_ROOT / 'reports' / 'day17_final_demo_cases.csv',
    keep_default_na=False,
)
test_set = pd.read_csv(
    PROJECT_ROOT / 'data' / 'interim' / 'test.csv',
    encoding='utf-8-sig',
    keep_default_na=False,
)
demo_order = [
    'likely_real_correct',
    'likely_fake_correct',
    'uncertain',
    'false_positive',
    'false_negative',
]
demo_notes = {
    'likely_real_correct': 'Thekso burimet e përmendura dhe formulimin institucional.',
    'likely_fake_correct': 'Thekso marker-in sensacional dhe mungesën e treguesve të burimit.',
    'uncertain': 'Shpjego se zona uncertain do të thotë mungesë sigurie, jo gjysmë e vërtetë.',
    'false_positive': 'Shpjego se stili paralajmërues mund të ngjajë me tekstet fake.',
    'false_negative': 'Shpjego bias-in e gjatësisë dhe stilin formal të një lajmi fake.',
}

demo_source = (
    frozen_demos[frozen_demos['demo_type'].isin(demo_order)][
        ['demo_type', 'article_id', 'true_label', 'decision',
         'probability_real', 'probability_fake']
    ]
    .merge(test_set[['article_id', 'title', 'content']], on='article_id', validate='one_to_one')
)
demo_source['demo_order'] = demo_source['demo_type'].map(
    {name: index + 1 for index, name in enumerate(demo_order)}
)

demo_rows = []
for row in demo_source.sort_values('demo_order').itertuples(index=False):
    result = predict_final_news(row.title, row.content, model=final_model)
    assert result['decision'] == row.decision
    assert np.isclose(result['probability_real'], row.probability_real)
    assert np.isclose(result['probability_fake'], row.probability_fake)
    signals = result['linguistic_explanation']
    demo_rows.append({
        'demo_order': row.demo_order,
        'demo_type': row.demo_type,
        'article_id': row.article_id,
        'true_label': row.true_label,
        'title': row.title,
        'content': row.content,
        'expected_decision': result['decision'],
        'probability_real': result['probability_real'],
        'probability_fake': result['probability_fake'],
        'word_count': signals['word_count'],
        'exclamation_count': signals['exclamation_count'],
        'uppercase_ratio': signals['uppercase_ratio'],
        'diacritic_ratio': signals['diacritic_ratio'],
        'sensational_markers': ' | '.join(signals['sensational_words_found']),
        'source_markers': ' | '.join(signals['source_markers_found']),
        'uncertainty_markers': ' | '.join(signals['uncertainty_markers_found']),
        'demo_note': demo_notes[row.demo_type],
    })

demo_cases = pd.DataFrame(demo_rows)
demo_output = PROJECT_ROOT / 'reports' / 'day19_demo_cases.csv'
demo_cases.to_csv(demo_output, index=False, encoding='utf-8-sig')
display(demo_cases[[
    'demo_order', 'demo_type', 'article_id', 'true_label',
    'expected_decision', 'probability_real', 'probability_fake', 'word_count',
]])
print(f'Rastet demo u ruajtën te: {demo_output.relative_to(PROJECT_ROOT)}')

## 6. Probabilitetet dhe zona `uncertain`

Vendimi përdor vetëm probabilitetin `P(fake)`: më i vogël se 0.30 është `likely_real`, nga 0.30 deri në 0.70 përfshirë është `uncertain`, dhe më i madh se 0.70 është `likely_fake`.

In [ ]:
thresholds = pd.DataFrame([
    {'decision': 'likely_real', 'rule': f'P(fake) < {FINAL_REAL_THRESHOLD:.2f}'},
    {'decision': 'uncertain', 'rule': f'{FINAL_REAL_THRESHOLD:.2f} <= P(fake) <= {FINAL_FAKE_THRESHOLD:.2f}'},
    {'decision': 'likely_fake', 'rule': f'P(fake) > {FINAL_FAKE_THRESHOLD:.2f}'},
])
display(thresholds)
display(demo_cases.loc[
    demo_cases['demo_type'] == 'uncertain',
    ['title', 'probability_real', 'probability_fake', 'expected_decision'],
])
display(Markdown(
    '**Interpretim:** `uncertain` tregon se modeli nuk ka siguri të mjaftueshme; '
    'nuk është pohim se lajmi është pjesërisht i vërtetë.'
))

## 7. Metrikat finale dhe confusion matrix

Metrikat lexohen nga output-i i ngrirë i Ditës 17. Notebook-u nuk i rillogarit për të zgjedhur model ose pragje.

In [ ]:
with (PROJECT_ROOT / 'reports' / 'day17_final_metrics.json').open(encoding='utf-8') as file:
    final_metrics = json.load(file)

internal = final_metrics['official_internal_metrics']
metric_names = [
    'accuracy', 'f1_weighted', 'f1_fake', 'recall_real', 'recall_fake',
    'brier_score', 'log_loss', 'ece', 'threshold_strong_coverage',
    'threshold_strong_accuracy',
]
display(pd.DataFrame({
    'metric': metric_names,
    'value': [internal[name] for name in metric_names],
}))

confusion = np.asarray(ast.literal_eval(internal['confusion_matrix']))
fig, ax = plt.subplots(figsize=(4.8, 3.8))
image = ax.imshow(confusion, cmap='Blues')
for row_index in range(confusion.shape[0]):
    for column_index in range(confusion.shape[1]):
        ax.text(column_index, row_index, confusion[row_index, column_index],
                ha='center', va='center', fontsize=12)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Predicted real', 'Predicted fake'],
       yticklabels=['True real', 'True fake'],
       title='Confusion matrix - test set i brendshëm')
ax.set_xlabel('Prediction')
ax.set_ylabel('Label')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 8. Vlerësimi i jashtëm pilot

Dataset-i i jashtëm ka 40 përmbledhje të shkurtra dhe domain shift të dukshëm. Ai u përdor vetëm për vlerësim pilot, jo për tuning ose zgjedhje të modelit.

In [ ]:
external = final_metrics['external_pilot_metrics']
comparison = pd.DataFrame([
    {
        'evaluation': 'internal_test',
        'articles': 792,
        'accuracy': internal['accuracy'],
        'recall_real': internal['recall_real'],
        'recall_fake': internal['recall_fake'],
        'brier_score': internal['brier_score'],
        'log_loss': internal['log_loss'],
    },
    {
        'evaluation': 'external_pilot',
        'articles': 40,
        'accuracy': external['accuracy'],
        'recall_real': external['recall_real'],
        'recall_fake': external['recall_fake'],
        'brier_score': external['brier_score'],
        'log_loss': external['log_loss'],
    },
])
display(comparison)

external_predictions = pd.read_csv(
    PROJECT_ROOT / 'reports' / 'day17_final_external_predictions.csv'
)
external_final = external_predictions[
    external_predictions['model'] == 'final_word_char_svm'
].copy()
topic_summary = (
    external_final.groupby('topic', as_index=False)
    .agg(
        articles=('external_id', 'count'),
        accuracy=('prediction_correct', 'mean'),
        mean_probability_fake=('probability_fake', 'mean'),
    )
)
display(topic_summary)

## 9. Shembull error analysis

Gabimet analizohen për të kuptuar kufizimet, jo për të ndryshuar modelin pas testimit. Shembulli më poshtë është false negative i përzgjedhur në Ditën 17.

In [ ]:
false_negative = demo_cases.loc[
    demo_cases['demo_type'] == 'false_negative'
].iloc[0]
display(pd.DataFrame([{
    'article_id': false_negative['article_id'],
    'title': false_negative['title'],
    'true_label': false_negative['true_label'],
    'decision': false_negative['expected_decision'],
    'probability_real': false_negative['probability_real'],
    'probability_fake': false_negative['probability_fake'],
    'word_count': false_negative['word_count'],
    'content_preview': false_negative['content'][:300],
}]))
display(Markdown(
    '**Interpretim i kujdesshëm:** artikulli fake është i gjatë dhe përdor stil formal, '
    'citime dhe emra institucionesh. Këto karakteristika mund ta afrojnë me tekstet real '
    'të corpus-it. Ky rast ilustron bias-in e gjatësisë dhe faktin se modeli klasifikon '
    'stil gjuhësor, jo vërtetësinë faktike.'
))

## 10. Përfundim

Rrjedha finale është: dataset → preprocessing NFC → Word + Character TF-IDF → Linear SVM → sigmoid calibration → probabilitete → pragjet me tri nivele → shpjegim gjuhësor.

Kufizimet kryesore janë domain shift-i, bias-i sipas gjatësisë, paqëndrueshmëria e teksteve shumë të shkurtra dhe mungesa e fact-checking-ut. Për ekzekutimin e aplikacionit dhe testet përdor komandat në `README.md`; kodi i plotë dhe i ripërdorshëm mbetet te `src/`.